> **ℹ️ Note**
>
**Durée estimée** : 4 à 5 heures  
**Prérequis** : Notions 3.1 (typologie) et 3.2 (stats descriptives)  
**Objectif** : maîtriser les techniques professionnelles de détection et traitement des anomalies dans un dataset


## 📋 Ce que tu sauras faire à la fin

- **Diagnostiquer** précisément les problèmes d'un dataset
- Détecter les **valeurs manquantes** avec discernement (MCAR, MAR, MNAR)
- Appliquer la **bonne stratégie d'imputation** selon le contexte
- Détecter et traiter les **outliers** avec plusieurs méthodes (IQR, z-score, percentiles)
- Gérer les **doublons** et les **incohérences** avec rigueur
- **Normaliser** les colonnes textuelles (capitalize, strip, regex)

## 🎯 Pourquoi c'est essentiel ?

Tu as appris les bases du nettoyage au Module 2. Mais en vrai, chaque dataset est un cas particulier, et les bonnes pratiques sont subtiles. Quelques exemples de questions que cette notion répond :

- Quand imputer par la moyenne, par la médiane, par le mode, ou par une méthode plus avancée ?
- Comment détecter un outlier sans méthode naïve ?
- Comment reconnaître un doublon "caché" (même client avec 2 orthographes différentes) ?
- Comment savoir si mes valeurs manquantes sont **aléatoires** ou **systématiques** ?

Ce sont ces questions qui séparent un data analyst débutant d'un data scientist expérimenté.

## 🛠️ Mise en route

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
np.random.seed(42)

print("✅ Environnement prêt")

# 1. Diagnostic complet d'un dataset

## 🔍 La checklist du nettoyage

Avant de commencer à nettoyer, tu dois **diagnostiquer**. Voici la checklist complète :

```
┌─────────────────────────────────────────┐
│  1. Shape et types de colonnes          │
│  2. Valeurs manquantes (NaN)            │
│  3. Doublons                            │
│  4. Outliers dans les variables quanti  │
│  5. Incohérences dans les valeurs       │
│  6. Textes non normalisés               │
│  7. Colonnes constantes ou quasi        │
└─────────────────────────────────────────┘
```

Créons un dataset volontairement **bien sale** pour illustrer :

In [ ]:
np.random.seed(42)
n = 300

dirty = pd.DataFrame({
    "id_client": range(1, n+1),
    "nom": np.random.choice(
        ["Alice Martin", "  alice martin ", "ALICE MARTIN", "Bob Dupont", "bob dupont"],
        size=n
    ),
    "age": np.random.randint(18, 85, n).astype(float),
    "salaire": np.random.normal(3500, 1000, n),
    "region": np.random.choice(["IDF", "  IDF ", "PACA", "paca", "ARA", None], n),
    "email": np.random.choice([
        "test@test.fr", "user@domain.com", "INVALID_EMAIL", "", None
    ], n),
    "nb_achats": np.random.poisson(5, n),
    "score_satisfaction": np.random.uniform(0, 10, n),
    "colonne_vide": [None] * n,  # colonne entièrement vide
    "pays": ["France"] * n       # colonne constante
})

# Injecter des pièges supplémentaires
dirty.loc[np.random.choice(n, 20), "age"] = np.nan               # NaN dispersés
dirty.loc[np.random.choice(n, 5), "age"] = 250                   # outliers
dirty.loc[np.random.choice(n, 10), "salaire"] = np.nan           # NaN
dirty.loc[0, "salaire"] = 1_000_000                              # outlier extrême
dirty.loc[1, "salaire"] = -500                                   # valeur incohérente
dirty.loc[np.random.choice(n, 8), "nb_achats"] = -1              # impossible
# Ajouter des doublons
doublons = dirty.sample(10, random_state=0)
dirty = pd.concat([dirty, doublons], ignore_index=True)

print(f"Dataset sale : {dirty.shape[0]} lignes x {dirty.shape[1]} colonnes")
dirty.head()

## 🧪 Le diagnostic automatique

Voici une fonction utile à **garder dans tes favoris** :

In [ ]:
def diagnostiquer(df):
    """Produit un rapport de diagnostic complet d'un DataFrame."""
    print("=" * 60)
    print(f"📊 DIAGNOSTIC — {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
    print("=" * 60)
    
    # 1. Valeurs manquantes
    nan_counts = df.isna().sum()
    nan_pct = (nan_counts / len(df) * 100).round(2)
    nan_report = pd.DataFrame({
        "NaN": nan_counts,
        "% NaN": nan_pct
    })
    nan_report = nan_report[nan_report["NaN"] > 0].sort_values("NaN", ascending=False)
    
    if len(nan_report) > 0:
        print("\n🕳️  VALEURS MANQUANTES")
        print(nan_report)
    else:
        print("\n✅ Aucune valeur manquante")
    
    # 2. Doublons
    n_doublons = df.duplicated().sum()
    print(f"\n🔁 DOUBLONS : {n_doublons}")
    
    # 3. Colonnes constantes
    const_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    if const_cols:
        print(f"\n📌 COLONNES CONSTANTES : {const_cols}")
    
    # 4. Types
    print("\n🏷️  TYPES")
    print(df.dtypes)
    
    # 5. Valeurs uniques pour les quali
    print("\n🔢 CARDINALITÉ")
    card = df.nunique().sort_values()
    print(card)

# Appliquer au dataset sale
diagnostiquer(dirty)

> **💡 Astuce**
>
## 🧠 Cette fonction, tu vas la réutiliser souvent
Copie-la dans un fichier `utils.py` personnel. Face à tout nouveau dataset, lance `diagnostiquer(df)` en premier — ça te donnera une vue d'ensemble en 2 secondes.


## ✏️ Exercice 1 — Diagnostic initial

> **ℹ️ Note**
>
## 📝 À faire

1. Lance `diagnostiquer(dirty)` et analyse le résultat.
2. Identifie **3 problèmes majeurs** à traiter en priorité.
3. Propose un **ordre de traitement** justifié.

In [ ]:
# TODO: Exercice 1 — diagnostic + plan d'action

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

# 2. Les valeurs manquantes : 3 types à distinguer

## 🎭 MCAR, MAR, MNAR

Les valeurs manquantes ne sont **pas toutes pareilles**. Les statisticiens distinguent 3 types :

### MCAR — Missing Completely At Random

Les NaN apparaissent **complètement au hasard**, sans aucun lien avec les autres variables ou avec la valeur elle-même.

**Exemple** : un capteur tombe en panne pendant une minute → quelques valeurs manquantes aléatoires dans un relevé météo.

**Conséquence** : pas de biais. Supprimer ou imputer → OK.

### MAR — Missing At Random

Les NaN dépendent d'**autres variables observées** mais pas de la valeur manquante elle-même.

**Exemple** : les hommes répondent moins à la question "pression artérielle" que les femmes → NaN dépend du genre mais pas de la pression.

**Conséquence** : on peut imputer correctement en utilisant les autres variables (imputation conditionnelle).

### MNAR — Missing Not At Random

Les NaN dépendent **de la valeur manquante elle-même**.

**Exemple** : les personnes aux revenus très élevés refusent de déclarer leur salaire.

**Conséquence** : c'est le pire cas. Imputer biaise le résultat. Il faut documenter et discuter de l'impact.

## 🔍 Visualiser les patterns de NaN

Avant d'imputer, regarde **où** sont les NaN. Ont-ils un pattern ?

In [ ]:
#| fig-cap: "Carte des valeurs manquantes"

# Créer un mini-dataset avec un pattern de NaN non-aléatoire
np.random.seed(0)
nm = pd.DataFrame({
    "age": np.random.randint(20, 70, 50),
    "salaire": np.random.normal(3000, 800, 50),
    "satisfaction": np.random.uniform(1, 10, 50),
})

# Les personnes jeunes ne répondent pas à satisfaction (MAR)
nm.loc[nm["age"] < 30, "satisfaction"] = np.where(
    np.random.random((nm["age"] < 30).sum()) < 0.7,
    np.nan, nm.loc[nm["age"] < 30, "satisfaction"]
)

# Carte thermique des NaN
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(nm.isna(), cbar=False, cmap="RdYlGn_r", ax=ax, 
            yticklabels=False)
ax.set_title("Carte des valeurs manquantes (rouge = NaN)")
plt.tight_layout()
plt.show()

Si tu vois un pattern (lignes consécutives de NaN, colonnes concentrées), ce n'est **pas MCAR** : il faut creuser.

## 🛠️ Les stratégies d'imputation

| Stratégie | Quand l'utiliser | Avantages | Inconvénients |
|---|---|---|---|
| **Supprimer les lignes** | Peu de NaN (< 5%), MCAR | Simple, pas de biais | Perte d'info |
| **Supprimer la colonne** | Beaucoup de NaN (> 50%) ou colonne peu importante | Rapide | Perte de variable |
| **Moyenne** | Quantitative, distribution symétrique | Simple | Réduit la variance |
| **Médiane** | Quantitative avec outliers | Robuste | Réduit la variance |
| **Mode** | Qualitative | Cohérent | Sur-représente la catégorie dominante |
| **Forward fill / backward fill** | Séries temporelles | Respecte la continuité | Ne marche que si ordre logique |
| **Imputation conditionnelle** | MAR, beaucoup de données | Plus précis | Plus complexe |
| **KNN imputer** | Relations complexes | Très précis | Coûteux, sensible aux outliers |

## 💡 Imputation simple

In [ ]:
df = dirty.copy()

# Supprimer la colonne entièrement vide
df = df.drop(columns=["colonne_vide", "pays"])

# Stratégies différentes selon la variable
# age : médiane (distribution possiblement asymétrique)
df["age"] = df["age"].fillna(df["age"].median())

# salaire : on impute par la médiane, mais on traite d'abord les outliers !
# (si on impute par la moyenne avec un outlier à 1M, la moyenne sera faussée)
df["salaire"] = df["salaire"].fillna(df["salaire"].median())

# region : mode
df["region"] = df["region"].fillna(df["region"].mode()[0])

# email : "Inconnu" (catégoriel avec beaucoup de cas particuliers)
df["email"] = df["email"].fillna("Inconnu")

print("NaN restants :")
print(df.isna().sum())

## 🧠 Imputation conditionnelle

Mieux que la médiane : **imputer selon un groupe**.

In [ ]:
# Exemple : imputer l'âge par la médiane DE LA RÉGION
# (si l'âge moyen varie par région, c'est plus précis)

# Recréer quelques NaN pour la démo
df_demo = dirty.copy()
df_demo["region"] = df_demo["region"].fillna("Inconnu")

# Imputation conditionnelle par groupe
df_demo["age_imputed"] = df_demo.groupby("region", observed=True)["age"].transform(
    lambda x: x.fillna(x.median())
)
print("✅ Imputation conditionnelle effectuée")
print(f"Age moyen imputé par région :")
print(df_demo.groupby("region", observed=True)["age_imputed"].median())

## 🚀 Imputation KNN (avancée)

Le **KNN imputer** de scikit-learn remplit chaque NaN en regardant les **k voisins les plus proches** (selon les autres variables).

In [ ]:
from sklearn.impute import KNNImputer

# On ne l'applique que sur les variables numériques
df_num = dirty[["age", "salaire", "nb_achats", "score_satisfaction"]].copy()

imputer = KNNImputer(n_neighbors=5)
df_num_imputed = pd.DataFrame(
    imputer.fit_transform(df_num),
    columns=df_num.columns
)

print(f"NaN avant : {df_num.isna().sum().sum()}")
print(f"NaN après : {df_num_imputed.isna().sum().sum()}")

> **⚠️ Attention**
>
## ⚠️ Le KNN imputer a des limites
- Il est **sensible aux outliers** → traiter les outliers AVANT de l'utiliser
- Il est **lent sur gros dataset** (> 100k lignes)
- Il nécessite **toutes les variables sur la même échelle** → normaliser avant (on verra ça Notion 3.4)


## ✏️ Exercice 2 — Imputation raisonnée

> **ℹ️ Note**
>
## 📝 À faire

Voici un dataset bancaire avec des valeurs manquantes :

```python
np.random.seed(0)
n = 200
banque = pd.DataFrame({
    "age": np.random.randint(18, 80, n).astype(float),
    "salaire": np.random.normal(3000, 1000, n),
    "nb_operations": np.random.poisson(20, n),
    "segment": np.random.choice(["Jeune", "Actif", "Senior"], n, p=[0.3, 0.5, 0.2])
})
# Les Seniors répondent moins sur le salaire (MAR)
# On prend les indices des Seniors, puis on en tire 60% à passer en NaN
idx_seniors = banque[banque["segment"] == "Senior"].index
idx_a_nuller = np.random.choice(idx_seniors, size=int(len(idx_seniors) * 0.6), replace=False)
banque.loc[idx_a_nuller, "salaire"] = np.nan

# Quelques NaN sur age (MCAR)
banque.loc[np.random.choice(n, 10, replace=False), "age"] = np.nan
```

1. Lance `diagnostiquer(banque)` et regarde les NaN.
2. Calcule le % de NaN par `segment` pour la colonne `salaire`. Est-ce MCAR ?
3. Impute `age` avec la **médiane globale** (c'est MCAR).
4. Impute `salaire` avec la **médiane par segment** (c'est MAR, mieux de conditionner).
5. Vérifie qu'il ne reste plus de NaN.

In [ ]:
# TODO: Exercice 2

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

# 3. Les outliers : détecter et traiter

## 🚨 4 méthodes de détection

### Méthode 1 — IQR (Interquartile Range)

Déjà vue au Module 1. La plus simple, robuste :

```
outlier si : valeur < Q1 - 1.5*IQR  OU  valeur > Q3 + 1.5*IQR
```

In [ ]:
def outliers_iqr(serie, coef=1.5):
    """Retourne un masque booléen des outliers."""
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    return (serie < q1 - coef*iqr) | (serie > q3 + coef*iqr)

# Sur notre dataset sale
outliers_salaire = outliers_iqr(dirty["salaire"])
print(f"Outliers salaire (IQR) : {outliers_salaire.sum()}")

### Méthode 2 — Z-score

L'outlier est à plus de `k` écarts-types de la moyenne (généralement `k=3`) :

```
z_i = (x_i - moyenne) / std
outlier si : |z_i| > 3
```

In [ ]:
def outliers_zscore(serie, seuil=3):
    """Détection par z-score."""
    z = (serie - serie.mean()) / serie.std()
    return z.abs() > seuil

outliers_z = outliers_zscore(dirty["salaire"])
print(f"Outliers salaire (z-score) : {outliers_z.sum()}")

> **⚠️ Attention**
>
## ⚠️ Z-score vs IQR
- **Z-score** suppose une distribution **normale**. Sur des données asymétriques, il donne des résultats trompeurs.
- **IQR** est plus **robuste** (ne fait pas d'hypothèse sur la distribution). C'est souvent le meilleur choix par défaut.


### Méthode 3 — Percentiles

Plus pragmatique : on considère comme outliers tout ce qui est au-delà des percentiles 1% et 99% (ou 5% et 95%).

In [ ]:
def outliers_percentiles(serie, low=1, high=99):
    """Détection par bornes percentiles."""
    borne_basse = serie.quantile(low / 100)
    borne_haute = serie.quantile(high / 100)
    return (serie < borne_basse) | (serie > borne_haute)

outliers_p = outliers_percentiles(dirty["salaire"], low=1, high=99)
print(f"Outliers salaire (1% / 99%) : {outliers_p.sum()}")

Cette méthode est simple et **prévisible** : tu sais à l'avance que tu retires X% des données.

### Méthode 4 — Validation métier

**La plus fiable** mais requiert du contexte : utilise les **règles du domaine**.

In [ ]:
# Pour le salaire : 
# - un salaire négatif est impossible
# - un salaire > 1 million est très suspect

outliers_metier = (dirty["salaire"] < 0) | (dirty["salaire"] > 500_000)
print(f"Outliers salaire (métier) : {outliers_metier.sum()}")

## 🛠️ Les 3 stratégies de traitement

Une fois détectés, **que faire des outliers** ?

### Stratégie 1 — Supprimer les lignes

In [ ]:
# Radical mais efficace
df_sans = dirty[~outliers_metier].copy()
print(f"Avant : {len(dirty)}, Après : {len(df_sans)}")

### Stratégie 2 — Capper (winsorizing)

Remplacer l'outlier par la borne acceptable :

In [ ]:
df_capped = dirty.copy()

# Borne supérieure : 99e percentile
borne_sup = df_capped["salaire"].quantile(0.99)
# Borne inférieure : minimum raisonnable (métier)
borne_inf = 0  # salaire ne peut pas être < 0

df_capped["salaire"] = df_capped["salaire"].clip(lower=borne_inf, upper=borne_sup)
print(f"Min salaire : {df_capped['salaire'].min():.0f}")
print(f"Max salaire : {df_capped['salaire'].max():.0f}")

### Stratégie 3 — Remplacer par NaN et imputer

Pour garder les lignes mais effacer la valeur erronée :

In [ ]:
df_replaced = dirty.copy()
mask = outliers_metier.reindex(df_replaced.index, fill_value=False)
df_replaced.loc[mask, "salaire"] = np.nan

# Puis imputer
df_replaced["salaire"] = df_replaced["salaire"].fillna(df_replaced["salaire"].median())
print(f"✅ {mask.sum()} outliers remplacés puis imputés")

## 📊 Visualiser l'impact du traitement

In [ ]:
#| fig-cap: "Avant / après traitement des outliers"

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Avant
salaire_clean = dirty["salaire"].dropna()
axes[0].boxplot(salaire_clean)
axes[0].set_title(f"Avant (N={len(salaire_clean)})")
axes[0].set_ylabel("Salaire")

# Après capping
salaire_capped = df_capped["salaire"].dropna()
axes[1].boxplot(salaire_capped)
axes[1].set_title(f"Après capping (N={len(salaire_capped)})")

plt.tight_layout()
plt.show()

## ✏️ Exercice 3 — Traiter les outliers

> **ℹ️ Note**
>
## 📝 À faire

Sur `dirty` :

1. Détecte les outliers de `age` avec la méthode **IQR**. Combien ?
2. Détecte les incohérences de `nb_achats` avec la **validation métier** (nb_achats < 0 est impossible). Combien ?
3. Sur `salaire` :
   - Compare le nombre d'outliers détectés par IQR, z-score et percentiles (1/99).
   - Quelle méthode est la plus conservative ?
4. Applique un **capping** à `age` : on garde entre 18 et 100 ans.
5. Remplace les `nb_achats` négatifs par 0 (valeur métier par défaut).

In [ ]:
# TODO: Exercice 3

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

# 4. Doublons : pas toujours évidents !

## 👥 Doublons parfaits vs doublons partiels

### Doublons parfaits

Toutes les colonnes sont identiques :

In [ ]:
n_parfaits = dirty.duplicated().sum()
print(f"Doublons parfaits : {n_parfaits}")

### Doublons partiels (souvent plus réels)

Même **clé métier** (ex: même client) mais avec des valeurs légèrement différentes :

In [ ]:
# Vérifier les doublons sur id_client
n_clients_dup = dirty.duplicated(subset=["id_client"]).sum()
print(f"Clients dupliqués (même id_client) : {n_clients_dup}")

## 🧬 Doublons "cachés" dus à un mauvais formatage

Le pire cas : **le même client avec 2 orthographes différentes**.

In [ ]:
# Sur la colonne 'nom' de notre dataset sale
print("Valeurs uniques de 'nom' (6 premières) :")
print(dirty["nom"].unique()[:6])

On voit `"Alice Martin"`, `"  alice martin "`, `"ALICE MARTIN"` — c'est **la même personne**.

## 🧹 Stratégies de déduplication

In [ ]:
# Stratégie : normaliser AVANT de dédupliquer
df = dirty.copy()

# 1. Normaliser le champ nom
df["nom_normalise"] = (
    df["nom"].str.strip()              # retire espaces en début/fin
             .str.lower()              # tout en minuscules
             .str.title()              # majuscules en début de mots
)

# Maintenant la déduplication devient possible
print("Valeurs uniques après normalisation :")
print(df["nom_normalise"].unique()[:6])

## 📅 Doublons sur plusieurs colonnes

In [ ]:
# Exemple : même commande (client + date + produit)
# Si 2 lignes ont les 3 mêmes valeurs, c'est un doublon métier

df_cmd = pd.DataFrame({
    "id_client": [1, 1, 2, 3, 1],
    "date": ["2024-01-01", "2024-01-01", "2024-01-02", "2024-01-03", "2024-01-01"],
    "produit": ["A", "A", "B", "C", "A"],
    "montant": [100, 100, 50, 200, 100],
})

# Déduplication sur (client, date, produit)
df_cmd_clean = df_cmd.drop_duplicates(subset=["id_client", "date", "produit"], keep="first")
print(f"Avant : {len(df_cmd)}, Après : {len(df_cmd_clean)}")

`keep="first"` garde la première occurrence. `keep="last"` garde la dernière. `keep=False` supprime **toutes** les occurrences de doublons (utile pour inspecter).

# 5. Normalisation textuelle

Même idée que pour les doublons : **les textes "sales" doivent être nettoyés** avant toute analyse.

## 🧴 Les 5 opérations de base

In [ ]:
# Un échantillon de textes sales
textes = pd.Series([
    "  Paris  ",
    "PARIS",
    "paris",
    "Pariis",
    "Marseille\t",
    "marseille ",
    "MARSEILLE",
    "lyon",
])

# 1. strip() : retire les espaces en début/fin
print("strip     :", textes.str.strip().unique())

# 2. lower() : tout en minuscules
print("\nlower     :", textes.str.lower().unique())

# 3. upper() : tout en majuscules
print("\nupper     :", textes.str.upper().unique())

# 4. title() : première lettre de chaque mot en majuscule
print("\ntitle     :", textes.str.title().unique())

# 5. replace() : remplacement classique
print("\nreplace   :", textes.str.replace(r'\s+', ' ', regex=True).unique())

## 🔤 Une normalisation complète

In [ ]:
def normaliser_texte(serie):
    """Normalise une série de textes de manière complète."""
    return (
        serie
        .astype(str)
        .str.strip()                         # retire espaces début/fin
        .str.replace(r'\s+', ' ', regex=True) # remplace multi-espaces par un seul
        .str.lower()                         # tout en minuscules
    )

# Application
textes_clean = normaliser_texte(textes)
print(f"Avant : {textes.nunique()} valeurs uniques")
print(f"Après : {textes_clean.nunique()} valeurs uniques")
print("Valeurs :", textes_clean.unique())

Note le gain : **8 valeurs "uniques" → 3 après normalisation**. C'est énorme.

## 🔍 Regex : pour les cas complexes

Les **expressions régulières** (regex) sont indispensables pour détecter des patterns dans du texte.

In [ ]:
# Exemple : valider des emails
emails = pd.Series([
    "user@domain.com",
    "INVALID_EMAIL",
    "other.user@example.fr",
    "wrong@",
    "@domain.com",
    "",
    "good.email+tag@company.co.uk"
])

# Pattern simple pour un email valide
pattern_email = r"^[\w\.\+-]+@[\w-]+\.[\w\.-]+$"
is_valid = emails.str.match(pattern_email, na=False)

# Afficher le résultat
resultat = pd.DataFrame({"email": emails, "valide": is_valid})
print(resultat)

## ✏️ Exercice 4 — Nettoyage textuel

> **ℹ️ Note**
>
## 📝 À faire

Sur `dirty` :

1. Normalise la colonne `nom` en une nouvelle colonne `nom_clean` (strip + title case).
2. Normalise la colonne `region` en une nouvelle colonne `region_clean` (strip + upper case).  
   *Attention aux valeurs NaN.*
3. Combien de valeurs uniques y a-t-il dans `region` **avant** et **après** normalisation ?
4. Sur la colonne `email`, crée une colonne booléenne `email_valide` qui vaut True si l'email matche un pattern valide.
5. Quel pourcentage d'emails est invalide ?

In [ ]:
# TODO: Exercice 4

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

# 🏁 Exercice bilan — Pipeline de nettoyage complet

> **ℹ️ Note**
>
## 📝 Énoncé

Tu vas écrire une **fonction de nettoyage** qui applique toutes les étapes apprises dans cet ordre, sur le dataset `dirty` :

1. Supprimer les colonnes inutiles (constantes ou 100% NaN)
2. Supprimer les doublons parfaits
3. Normaliser les textes (`nom`, `region`)
4. Capper les outliers : `age` entre 18 et 100, `salaire` entre 0 et le 99e percentile
5. Remplacer les `nb_achats` négatifs par 0
6. Imputer les NaN restants :
   - `age` → médiane
   - `salaire` → médiane
   - `region` → "Inconnue"
7. Retourner le DataFrame nettoyé + un **rapport** (dictionnaire) avec le nombre d'actions faites à chaque étape

La fonction doit :
- Prendre `df` en entrée
- **Ne pas modifier l'original** (toujours faire `df.copy()`)
- Retourner un tuple `(df_clean, rapport)`

Appelle-la sur `dirty` et affiche le rapport.

In [ ]:
# TODO: Exercice bilan — pipeline complet

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

# 🎓 Récapitulatif

## 📋 La checklist du nettoyage

Face à un nouveau dataset, **toujours** dans cet ordre :

```
☐ 1. Diagnostic (fonction diagnostiquer)
☐ 2. Suppression des colonnes inutiles
☐ 3. Suppression des doublons
☐ 4. Normalisation des textes
☐ 5. Correction des incohérences métier
☐ 6. Détection et traitement des outliers
☐ 7. Imputation des NaN
☐ 8. Vérification finale
```

## 🧠 Les arbitrages à maîtriser

| Question | Réponse par défaut |
|---|---|
| Supprimer ou imputer les NaN ? | **Imputer** si < 30% de NaN |
| Moyenne ou médiane ? | **Médiane** si distribution asymétrique ou outliers |
| Quelle méthode pour les outliers ? | **IQR** par défaut, **percentiles** pour précision |
| Supprimer ou capper les outliers ? | **Capper** si tu veux garder les lignes |
| Imputer globalement ou par groupe ? | **Par groupe** si tu détectes un pattern (MAR) |

## 🚨 Les pièges à éviter

1. **Imputer avant de traiter les outliers** → la moyenne/médiane sera biaisée
2. **Déduplicater avant de normaliser** → tu loupes les "vrais" doublons
3. **Utiliser le z-score sur une distribution asymétrique** → faux positifs
4. **Ne pas documenter** ses choix de nettoyage → non reproductible
5. **Appliquer aveuglément le KNN imputer** → sensible aux outliers

## 🚀 La suite

Dans la [**Notion 3.4 — Préparation pour le ML**](notion_3_4_preparation_ml.qmd), on va apprendre à **transformer** les données nettoyées pour les rendre compatibles avec des algorithmes de machine learning :

- Encoding (label, one-hot, ordinal)
- Scaling (min-max, standard)
- Extraction de features à partir de dates
- Train/test split propre

C'est le **dernier maillon** avant le Module 4 (Machine Learning supervisé).

> **💡 Astuce**
>
## 💡 Défi pour consolider
Enrichis ta fonction `nettoyer_dataset()` pour qu'elle soit plus flexible :

- Permettre de choisir la stratégie d'imputation (moyenne / médiane / mode)
- Accepter un dictionnaire de règles métier (`{"salaire": {"min": 0, "max": 500000}}`)
- Produire un log détaillé (qui, quand, quoi)

Mets-la dans un fichier `utils.py` personnel — tu vas la réutiliser sur tous tes futurs projets.